In [45]:
import spacy
from spacy.tokens import DocBin
from tqdm import tqdm
from spacy.util import filter_spans

import os
import sys
import dotenv

import json
from pathlib import Path

from collections import defaultdict
from prettytable import PrettyTable

dotenv.load_dotenv()
ROOT_DIR = os.environ.get("ROOT_DIR")
sys.path.append(f"{ROOT_DIR}/scripts")

from evaluation import evaluate

In [12]:
test_before_2000 = json.load(open(f"{ROOT_DIR}/data/splits/test_before_2000.json", "r"))
test_after_2000 = json.load(open(f"{ROOT_DIR}/data/splits/test_after_2000.json", "r"))

# Spacy without training

# Inference

In [13]:
nlp = spacy.load("fr_core_news_sm")

In [14]:
def inference(nlp, data):
    all_doc = []
    for doc in data:
        entities = []
        text = doc["texte"]
        doc["predicted_entities"] = []
        spacy_doc = nlp(text)
        for ent in spacy_doc.ents:
            entities.append({
                "texte": ent.text,
                "tag": ent.label_,
                "debut": ent.start_char,
                "fin": ent.end_char})

        all_doc.append({
            "id": doc["id"],
            "annee": doc["annee"],
            "predicted_entities": entities
        })

    return all_doc

In [15]:
inference_before_2000 = inference(nlp, test_before_2000)
inference_after_2000 = inference(nlp, test_after_2000)   

# Evaluation

In [16]:
metrics_before_2000 = evaluate(inference_before_2000, test_before_2000)

Global NER Performance (Exact Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.0775 |
|   Recall  | 0.2448 |
|  F1-Score | 0.1177 |
+-----------+--------+

Global NER Performance (Partial Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.1572 |
|   Recall  | 0.4965 |
|  F1-Score | 0.2388 |
+-----------+--------+

Performance by Tag — Exact Match (MISC excluded)
+------+-----------+--------+----------+---------+------------+
| Tag  | Precision | Recall | F1-Score | Support | Partial TP |
+------+-----------+--------+----------+---------+------------+
| LOC  |   0.0571  | 0.2346 |  0.0918  |    81   |     34     |
| MISC |   0.0096  | 0.0488 |  0.0161  |    82   |     8      |
| ORG  |   0.1019  | 0.2132 |  0.1379  |   197   |     42     |
| PER  |   0.2062  | 0.5797 |  0.3042  |    69   |     24     |
+------+-----------+--------+----------+---------+------------+


In [17]:
metrics_after_2000 = evaluate(inference_after_2000, test_after_2000)

Global NER Performance (Exact Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.0260 |
|   Recall  | 0.4545 |
|  F1-Score | 0.0491 |
+-----------+--------+

Global NER Performance (Partial Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.0349 |
|   Recall  | 0.6116 |
|  F1-Score | 0.0661 |
+-----------+--------+

Performance by Tag — Exact Match (MISC excluded)
+------+-----------+--------+----------+---------+------------+
| Tag  | Precision | Recall | F1-Score | Support | Partial TP |
+------+-----------+--------+----------+---------+------------+
| LOC  |   0.0398  | 0.5000 |  0.0738  |    42   |     6      |
| MISC |   0.0000  | 0.0000 |  0.0000  |    10   |     1      |
| ORG  |   0.0401  | 0.7692 |  0.0762  |    26   |     5      |
| PER  |   0.0308  | 0.3256 |  0.0563  |    43   |     7      |
+------+-----------+--------+----------+---------+------------+


# Trained Spacy

In [18]:
training_data_raw = json.load(open(f"{ROOT_DIR}/data/splits/train.json", "r"))

In [23]:
training_data = {'classes' : ['PER', 'ORG', 'MISC', 'LOC'], 'annotations' : []}
for doc in training_data_raw:
  temp_dict = {}
  temp_dict['texte'] = doc['texte']
  temp_dict['entites'] = []
  for annotation in doc['entites']:
    debut = annotation['debut']
    fin = annotation['fin']
    tag = annotation['tag'].upper()
    temp_dict['entites'].append((debut, fin, tag))
  training_data['annotations'].append(temp_dict)
  
print(training_data['annotations'][0]['entites'])

[(0, 11, 'ORG'), (14, 21, 'ORG'), (82, 95, 'PER'), (113, 116, 'LOC'), (117, 136, 'LOC'), (138, 153, 'LOC'), (187, 223, 'ORG'), (246, 279, 'ORG'), (394, 405, 'LOC'), (535, 553, 'LOC'), (1001, 1019, 'PER'), (1227, 1233, 'LOC'), (1939, 1945, 'LOC'), (2086, 2092, 'LOC'), (2180, 2191, 'LOC'), (2292, 2298, 'LOC'), (2419, 2425, 'LOC'), (2499, 2505, 'LOC'), (2522, 2528, 'LOC'), (2734, 2759, 'MISC'), (2760, 2778, 'MISC'), (2779, 2793, 'MISC'), (2834, 2845, 'MISC'), (2846, 2859, 'MISC'), (2854, 2858, 'LOC'), (2860, 2918, 'MISC'), (2972, 2985, 'LOC'), (1191, 1202, 'ORG'), (1215, 1222, 'ORG')]


## Training

In [24]:
# 
nlp = spacy.blank("fr") # créer un pipeline vide pour le français, c'est juste un tokenizer
doc_bin = DocBin() # create a DocBin object

In [71]:
for training_example  in tqdm(training_data['annotations']): 
    text = training_example['texte']
    labels = training_example['entites']
    doc = nlp.make_doc(text) 
    ents = []
    for debut, fin, tag in labels:
        span = doc.char_span(debut, fin, label=tag, alignment_mode="contract")
        # if span is None:
        #     print("Skipping entity")
        if span is not None:
            ents.append(span)
    filtered_ents = filter_spans(ents)
    doc.ents = filtered_ents 
    doc_bin.add(doc)

doc_bin.to_disk(f"{ROOT_DIR}/data/spacy/training_data.spacy") # save the docbin object

100%|██████████| 46/46 [00:00<00:00, 273.30it/s]


In [76]:
# Générer le fichier de configuration pour l'entraînement

!python -m spacy init fill-config base_config.cfg config.cfg

✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [77]:
BASE_DIR = Path().resolve()

train_path = BASE_DIR / ".." / "data" / "spacy" / "training_data.spacy"
output_path = BASE_DIR / ".." / "models"

print(train_path.resolve())

/Users/wiamlachqer/Desktop/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/data/spacy/training_data.spacy


In [78]:
!python -m spacy init config config.cfg --lang fr --pipeline ner --force

⚠ To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
ℹ Generated config template specific for your use case
- Language: fr
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [80]:
!python -m spacy train config.cfg \
  --output {output_path} \
  --paths.train {train_path} \
  --paths.dev {train_path} \
  --training.optimizer.learn_rate=0.001 \
  --training.max_epochs=10 \
  --training.max_steps=0

ℹ Saving to output directory:
/Users/wiamlachqer/Desktop/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/notebooks/../models
ℹ Using CPU
ℹ To switch to GPU 0, use the option: --gpu-id 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00    608.50    0.00    0.00    0.00    0.00
  0     200       1013.10  14814.76   16.28   91.00    8.94    0.16
  0     400       1769.77   4733.19   57.39   71.23   48.05    0.57
  0     600       1182.89   3791.43   63.26   64.99   61.62    0.63
  1     800        961.20   3404.47   69.27   75.77   63.80    0.69
  1    1000       1212.23   2702.80   76.22   78.78   73.83    0.76
  1    1200       1778

## Evaluation

In [ ]:
nlp_ner = spacy.load("model-best")

In [ ]:
trained_inference_before_2000 = inference(nlp, test_before_2000)
trained_inference_after_2000 = inference(nlp, test_after_2000) 